# MHCflurry CPU example
Peptide–MHC Class I Binding and Presentation Prediction Model
## Overview
- **Field** — Immunogenicity prediction
- **Category** — Immunology
- **Model type** — Neural network (ensemble)
- **Year** — 2020
- **License** — Apache-2.0
**Related papers**

- [MHCflurry 2.0: Improved Pan-Allele Prediction of MHC Class I-Presented Peptides by Incorporating Antigen Processing](https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30239-8) — Timothy J. O'Donnell et al., Cell Systems, 2020
- [MHCflurry: Open-Source Class I MHC Binding Affinity Prediction](https://www.cell.com/cell-systems/fulltext/S2405-4712(18)30232-1) — Timothy J. O'Donnell et al., Cell Systems, 2018

> **CPU example.** This notebook uses a single SMILES or a short peptide input. Its input workflow, package version, and output types may differ from the platform.

**Notebook package version:** 2.3.16. **Platform catalog version:** 1.0.0. Do not assume predictions from different versions are equivalent. Installed versions are recorded with the outputs.

## Requirements

- **Repository** — https://github.com/openvax/mhcflurry
- **Getting the weights** — After installing, run `mhcflurry-downloads fetch` to pull the pretrained weights.
- **Libraries** — see the environment section below
- **Run it on the platform** — https://lab.curieus.net/Analysis/mhcflurry

## Limits and cautions
- Runs on CPU — no GPU runtime needed.
- Downloading the weights takes several minutes.
- When the session ends, installed packages and downloaded weights are gone. Reconnecting means installing again.
- The free runtime disconnects after a period of inactivity. For long runs, saving intermediate results to Google Drive is safer.

## Procedure

Use a CPU runtime and run the setup, input, inference, and results cells in order. A GPU is not required.

This example predicts MHC binding and presentation-related scores for two short peptides. Its scope differs from the platform's full FASTA scanning workflow.

## Set up the environment

In [ ]:
import os, sys, platform, subprocess
from pathlib import Path
from sysconfig import get_path
SCRIPTS = Path(get_path("scripts"))
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # CPU example
os.environ['MPLBACKEND'] = 'Agg'  # CLI runs outside the notebook kernel
print('Python:', platform.python_version())
subprocess.run([sys.executable, "-m", "pip", "install", "mhcflurry==2.3.16"], check=True)

## Set the inputs

In [ ]:
ALLELE = "HLA-A*02:01"  # @param {type:"string"}
PEPTIDES = ["SIINFEKL", "SYNFEKKL"]
subprocess.run([str(SCRIPTS / "mhcflurry-downloads"), "fetch", "models_class1_presentation"], check=True, timeout=1200)

## Run

In [ ]:
from pathlib import Path
Path("preds.csv").unlink(missing_ok=True)
subprocess.run([str(SCRIPTS / "mhcflurry-predict"), "--alleles", ALLELE, "--peptides", *PEPTIDES, "--out", "preds.csv"], check=True, timeout=600)

## Check the results

In [ ]:
import json
import pandas as pd
from importlib.metadata import version
from IPython.display import display
result_path = Path("preds.csv")
if not result_path.is_file() or result_path.stat().st_size == 0:
    raise RuntimeError("Prediction output was not created")
predictions = pd.read_csv(result_path)
expected_rows = len(PEPTIDES)
if len(predictions) != expected_rows or predictions.select_dtypes(include="number").dropna(how="all").empty:
    raise RuntimeError("Prediction output is empty or incomplete")
required_scores = ["mhcflurry_affinity","mhcflurry_presentation_score"]
import numpy as np
if not set(required_scores).issubset(predictions.columns):
    raise RuntimeError("Expected model prediction columns are missing")
scores = predictions[required_scores].apply(pd.to_numeric, errors="raise")
if not np.isfinite(scores.to_numpy()).all():
    raise RuntimeError("Model prediction scores contain missing or non-finite values")
display(predictions)
environment = {"python": platform.python_version(), "package": "mhcflurry", "version": version("mhcflurry"), "platform_catalog_version": "1.0.0"}
Path("environment.json").write_text(json.dumps(environment, indent=2))
print(environment)
print("Saved: preds.csv, environment.json")

## Interpreting the results

The output is a per-peptide prediction CSV. Lower affinity values indicate stronger predicted binding. Presentation scores are a separate metric and do not establish experimental immunogenicity.

Consult the paper and repository for metric definitions.